In [1]:
# add the path to src2 to the sys.path
import sys
sys.path.append('/staging/users/tpadhi1/Multimodal-Uncertainty-Quantification/src2')
import torch
from PIL import Image
import os
# from inference_utils import generate_explanations_MM
from data_utils import get_vqa_dataloader, VQADataset, vqa_collate_fn
from torch.utils.data import DataLoader
from transformers import AutoProcessor, LlavaForConditionalGeneration

# Updated generate_explanations_MM without batching
def generate_explanations_MM(model, processor, sample, rank, params):
    try:
        torch_device = f'cuda:{rank}' if torch.cuda.is_available() else 'cpu'
        print(f'Using device: {torch_device}')
        
        raw_image = Image.open(sample['image_paths'][0])
        print('Image loaded successfully.')
        
        question_id = sample['question_ids'][0]
        print(f'Question ID: {question_id}')
    
        print(f"Generating explanations for question {question_id}")
    
        total = params['no_of_responses_sampled_per_image']
        print(f'Total responses: {total}')
    
        for i in range(total):
            prompt = sample['promptified_questions'][0]
            image = raw_image
            print(f'Processing response {i + 1}/{total}')
        
            inputs = processor(
                images=image, 
                text=prompt, 
                return_tensors="pt", 
                padding=True, 
                truncation=True
            ).to(torch_device)
            print('Inputs processed and moved to device.')
        
            outputs = model.generate(
                **inputs,
                do_sample=True,
                temperature=params['temperature'],
                top_p=params['top_p'],
                num_beams=params['num_beams'],
                max_new_tokens=params['max_new_tokens'],
                use_cache=False,
                return_dict_in_generate=True,
                output_scores=True
            )
            print('Model generation completed.')
    
            # generated_token_ids = outputs.sequences[0, inputs['input_ids'].shape[-1]:]
            # i am getting empty responses, so trying to get the all the token ids
            generated_token_ids = outputs.sequences[0]
            decoded_outputs = processor.decode(generated_token_ids, skip_special_tokens=True).split('\n')[0]
            transition_scores = model.compute_transition_scores(outputs.sequences, outputs.scores, normalize_logits=True)
    
            sample[f'response_{i}'] = {
                'prompt': prompt,
                'transition_scores': transition_scores[0].cpu().numpy(),
                'generated_token_ids': generated_token_ids.cpu().numpy(),
                'decoded_outputs': decoded_outputs
            }
            print(f'Response {i} generated : ', decoded_outputs)
    
        del inputs, outputs, generated_token_ids, decoded_outputs, transition_scores
        torch.cuda.empty_cache()
        print('Cache cleared.')
        
        return sample
    
    except Exception as e:
        print(f"An error occurred: {e}")

/home/tpadhi1/miniconda3/envs/llava/lib/python3.10/site-packages/transformers/utils/generic.py:482: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/tpadhi1/miniconda3/envs/llava/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/home/tpadhi1/miniconda3/envs/llava/lib/python3.10/site-packages/transformers/utils/generic.py:339: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/tpadhi1/miniconda3/envs/llava/lib/python3.10/site-packages/transformers/utils/generic.py:339: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.registe

In [2]:
# Initialize model and processor
model_id_llava = "llava-hf/llava-1.5-7b-hf"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_llava = LlavaForConditionalGeneration.from_pretrained(model_id_llava).to(device)
processor_llava = AutoProcessor.from_pretrained(model_id_llava)



Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some kwargs in processor config are unused and will not have any effect: num_additional_image_tokens. 


In [3]:
config_data = {
    "dataset": "vqa",
    "root_dir": "/staging/users/tpadhi1/Multimodal-Uncertainty-Quantification/datasets_/VQA",
    "image_dir": "train2014",
    "annotation_file": "v1/annotations/mscoco_train2014_annotations_HIGH_agreement_question_types_GROUND.json",
    "question_file": "v1/questions/OpenEnded_mscoco_train2014_questions_RANDOMSEED_42_COUNT_2000_CURRDATE_20241228_170848.json",
    "dataset_type": "json",
    "samples": 2000
}


# Create dataloader
dataloader = DataLoader(VQADataset(config_data), batch_size=1, collate_fn=vqa_collate_fn)

# Get a sample
sample = next(iter(dataloader))

rank = 0


Number of annotations: 25058
Number of questions: 2000


In [4]:
params = {
    # 'inference_batch_size': 20,  # Not used since batching is removed
    'no_of_responses_sampled_per_image': 20,
    'temperature': 1.0,
    'top_p': 0.9,
    'num_beams': 5,
    'max_new_tokens': 50
}
config_logging = {
    'explanation_dir': '/staging/users/tpadhi1/Multimodal-Uncertainty-Quantification/explanations'
}

sample_output = generate_explanations_MM(model_llava, processor_llava, sample, rank, params)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Using device: cuda:0
Image loaded successfully.
Question ID: 3823162
Generating explanations for question 3823162
Total responses: 20
Processing response 1/20
Inputs processed and moved to device.
Model generation completed.
Response 0 generated :  USER: Answer the questions. Here are few examples:
Processing response 2/20
Inputs processed and moved to device.
Model generation completed.
Response 1 generated :  USER: Answer the questions. Here are few examples:
Processing response 3/20
Inputs processed and moved to device.
Model generation completed.
Response 2 generated :  USER: Answer the questions. Here are few examples:
Processing response 4/20
Inputs processed and moved to device.


KeyboardInterrupt: 

In [8]:
# get all the values in sample data
print(sample.keys())
# dict_keys(['question_ids', 'questions', 'promptified_questions', 'answers', 'all_answers', 'image_ids', 'image_paths', 'response_0', 'response_1', 'response_2'])

# print()
print("promptified_question:", sample['promptified_questions'][0])
print("image path:", sample['image_paths'][0])



dict_keys(['question_ids', 'questions', 'promptified_questions', 'answers', 'all_answers', 'image_ids', 'image_paths', 'response_0', 'response_1', 'response_2'])
promptified_question: USER: Answer the questions. Here are few examples:
        Question: What is the color of the object?
        Answer: The color of the object is red.
        Question: What are the people doing ?
        Answer: The people in the image are playing soccer.
        Question: What animal is in the image?
        Answer: The animal is a cat.
        
        <image>
        What is the man carrying?
        ASSISTANT: 
        
image path: /staging/users/tpadhi1/Multimodal-Uncertainty-Quantification/datasets_/VQA/train2014/COCO_train2014_000000382316.jpg


In [19]:
# prompt = """
# USER:   <image>
# What is the man carrying?
# ASSISTANT: 
# """
prompt = """ USER: Answer the questions. Here are few examples:
        Question: What is the color of the object?
        Answer: The color of the object is red.
        Question: What are the people doing ?
        Answer: The people in the image are playing soccer.
        Question: What animal is in the image?
        Answer: The animal is a cat.
        
        <image>
        What is the man carrying?
        ASSISTANT: 
        """
image_path = sample['image_paths'][0]
image = Image.open(image_path)

inputs = processor_llava(
    images=image, 
    text=prompt, 
    return_tensors="pt", 
    padding=True, 
    truncation=True
).to(device)
print('Inputs processed and moved to device.')

# for i in range(20):
outputs = model_llava.generate(
    **inputs,
    do_sample=True,
    temperature=1.0,
    top_p=params['top_p'],
    num_beams=params['num_beams'],
    max_new_tokens=params['max_new_tokens'],
    use_cache=False,
    return_dict_in_generate=True,
    output_scores=True
)

Inputs processed and moved to device.


In [20]:
# now we can get the generated token ids
generated_token_ids = outputs.sequences[0]
decoded_outputs = processor_llava.decode(generated_token_ids, skip_special_tokens=True)

print("Generated response:", decoded_outputs)
# got empty response for temperature 1.0, so trying with 0.5
# with temperature 0.5, got the response
# The man is carrying a bicycle and an umbrella.
# so I tried with temperature 1.0, also i got the response, but here the prompt is different, it doesn't have the ICL examples
# so I tried the prompt with ICL examples, and got the response, but temperature 0.5 is giving the response, so lets try with 1.0
# so I tried with temperature 1.0, also i got the response, with ICL examples in the prompt

Generated response:  USER: Answer the questions. Here are few examples:
        Question: What is the color of the object?
        Answer: The color of the object is red.
        Question: What are the people doing ?
        Answer: The people in the image are playing soccer.
        Question: What animal is in the image?
        Answer: The animal is a cat.
        
         
        What is the man carrying?
        ASSISTANT: 
        
        The man is carrying a bicycle and an umbrella.
